In [1]:
#drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install --upgrade ultralytics opencv-python pandas
!pip install torch torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!git clone https://github.com/princeton-vl/RAFT.git /content/RAFT

Cloning into '/content/RAFT'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 147 (delta 0), reused 0 (delta 0), pack-reused 146 (from 2)
Receiving objects: 100% (147/147), 10.01 MiB | 29.47 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [4]:
!pip install pillow matplotlib tensorboardx scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.3 MB/s eta 0:00:00


In [9]:
# 전체 객체 위험도 기반 경고 시스템 코드 (클래스 매칭 및 obj_score 보정 포함)

from google.colab.patches import cv2_imshow
import cv2
import time
import numpy as np
import torch
from ultralytics import YOLO
import sys
import argparse
import pandas as pd

sys.path.append('/content/RAFT/core')
from raft import RAFT
from utils.utils import InputPadder

# RAFT Optical Flow 설정
RAFT_CKPT = '/content/drive/MyDrive/25_ESD/raft-things.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_raft_args():
    args = argparse.Namespace()
    args.small = False
    args.mixed_precision = False
    args.alternate_corr = False
    return args

raft_args = get_raft_args()
raft_model = RAFT(raft_args)
state_dict = torch.load(RAFT_CKPT, map_location=device)
if "module" in list(state_dict.keys())[0]:
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        new_key = k.replace("module.", "")
        new_state_dict[new_key] = v
    state_dict = new_state_dict
raft_model.load_state_dict(state_dict)
raft_model = raft_model.eval().to(device)

def estimate_flow_raft(prev_img, curr_img, model):
    img1 = cv2.cvtColor(prev_img, cv2.COLOR_BGR2RGB)
    img2 = cv2.cvtColor(curr_img, cv2.COLOR_BGR2RGB)
    t1 = torch.from_numpy(img1).permute(2,0,1).unsqueeze(0).float().to(device) / 255.
    t2 = torch.from_numpy(img2).permute(2,0,1).unsqueeze(0).float().to(device) / 255.
    padder = InputPadder(t1.shape)
    t1, t2 = padder.pad(t1, t2)
    with torch.no_grad():
        _, flow = model(t1, t2, iters=20, test_mode=True)
    return flow[0].permute(1,2,0).cpu().numpy()

# 클래스 ID 설정 (COCO 기준)
ALLOWED_CLASSES = {0, 1, 2, 3, 5, 6, 7}
SPEED_THRESHOLDS = {
    0: 3.0,  # person
    1: 5.0,  # bicycle
    2: 7.0,  # car
    3: 5.0,  # motorcycle
    5: 7.0,  # bus
    6: 7.0,  # train
    7: 7.0   # truck
}

FRAME_SIZE = 480
prev_info = {}
alert_cooldowns = {}
alerts_log = []
frame_idx = 0
top_dist_m = 4.0
bottom_dist_m = 2.5

def should_alert(obj_id, risk_score, t_now):
    if risk_score >= 10: return True
    cooldown = 2 if risk_score >= 6 else 5
    last_alert = alert_cooldowns.get(obj_id, 0)
    if t_now - last_alert > cooldown:
        alert_cooldowns[obj_id] = t_now
        return True
    return False

def compute_risk_score(speed, distance, obj_class_name, approach_angle):
    if approach_angle < 0.5:
        dist_score = max(0, 5 - distance)
    else:
        dist_score = 2.0 if distance < 2.5 else 0.0
    speed_score = min(speed / 2.0, 5.0)
    obj_score = {
        'car': 3,
        'bus': 3,
        'truck': 3,
        'bicycle': 1,
        'motorcycle': 2,
        'person': 0,
        'train': 3
    }.get(obj_class_name, 1)
    return dist_score + speed_score + obj_score

model = YOLO("/content/drive/MyDrive/25_ESD/yolo11x.pt")
cap = cv2.VideoCapture("/content/drive/MyDrive/25_ESD/depth_estimation/20250603_160952.mp4")
prev_time = time.time()

top_y = int(FRAME_SIZE * 0.55)
bottom_y = int(FRAME_SIZE * 0.95)
center_x = FRAME_SIZE // 2
top_width, bottom_width = 100, 220
zone_pts = np.array([
    (center_x - top_width//2, top_y),
    (center_x + top_width//2, top_y),
    (center_x + bottom_width//2, bottom_y),
    (center_x - bottom_width//2, bottom_y)
], dtype=np.int32)

prev_frame = None
output_path = '/content/drive/MyDrive/25_ESD/alert_system_result.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(output_path, fourcc, 20, (FRAME_SIZE, FRAME_SIZE))

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_resized = cv2.resize(frame, (FRAME_SIZE, FRAME_SIZE))

    affine_matrix = None
    if prev_frame is not None:
        flow = estimate_flow_raft(prev_frame, frame_resized, raft_model)
        h, w = flow.shape[:2]
        step = 30
        y_grid, x_grid = np.mgrid[0:h:step, 0:w:step]
        pts1 = np.stack([x_grid.ravel(), y_grid.ravel()], axis=1).astype(np.float32)
        pts2 = pts1 + flow[y_grid, x_grid].reshape(-1, 2)
        if len(pts1) >= 6:
            affine_matrix, _ = cv2.estimateAffinePartial2D(pts1, pts2)
    prev_frame = frame_resized.copy()

    results = model.track(source=frame_resized, persist=True, stream=True, tracker="botsort.yaml")
    t_now = time.time()

    cv2.polylines(frame_resized, [zone_pts], isClosed=True, color=(0,255,0), thickness=2)
    zone_center = tuple(np.mean(zone_pts, axis=0).astype(int))
    for r in results:
        for box in r.boxes:
            if not hasattr(box, 'id') or box.id is None:
                continue
            obj_id = int(box.id.item())
            cls = int(box.cls.item())
            if cls not in ALLOWED_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            bottom_center = (cx, y2)

            if obj_id in prev_info:
                px, py, t_prev, prev_speed, prev_warn_candidate = prev_info[obj_id]
                dt = t_now - t_prev
                if affine_matrix is not None:
                    prev_pt = np.array([[px, py]], dtype=np.float32)
                    corrected_pt = cv2.transform(prev_pt[None, :, :], affine_matrix)[0][0]
                    dx, dy = cx - corrected_pt[0], cy - corrected_pt[1]
                else:
                    dx, dy = cx - px, cy - py
                dist = np.hypot(dx, dy)
                if dt < 0.02 or dist > 100:
                    speed = 0
                else:
                    speed = dist / dt
                    if abs(speed - prev_speed) > 30:
                        speed = prev_speed
            else:
                speed, prev_warn_candidate, dist = 0, False, 0
                dx, dy = 0, 0

            approach_angle = abs(dx) / (abs(dy) + 1e-6)
            in_zone = cv2.pointPolygonTest(zone_pts, bottom_center, False) >= 0
            speed_thresh = SPEED_THRESHOLDS.get(cls, 1.5)
            current_warn_candidate = in_zone and speed > speed_thresh

            warn = False
            risk_score = None
            if current_warn_candidate and prev_warn_candidate:
                distance_m = ((y2 - top_y) / (bottom_y - top_y)) * (bottom_dist_m - top_dist_m) + top_dist_m
                risk_score = compute_risk_score(speed, distance_m, model.names[cls], approach_angle)
                if should_alert(obj_id, risk_score, t_now):
                    warn = True
                    alerts_log.append({
                        "frame": frame_idx,
                        "id": obj_id,
                        "class": model.names[cls],
                        "speed": speed,
                        "distance_m": distance_m,
                        "risk_score": risk_score,
                        "cx": cx,
                        "cy": cy,
                        "y2": y2,
                        "in_zone": int(in_zone)
                    })

            prev_info[obj_id] = (cx, cy, t_now, speed, current_warn_candidate)
            label = f"{model.names[cls]} {speed:.1f}px/s"
            color = (0, 0, 255) if warn else (255, 255, 0)
            if warn:
                label += " ⚠"
            cv2.rectangle(frame_resized, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame_resized, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            cv2.line(frame_resized, bottom_center, zone_center, (0, 255, 255), 2)

    curr_time = time.time()
    fps = 1.0 / (curr_time - prev_time + 1e-6)
    prev_time = curr_time
    cv2.putText(frame_resized, f"FPS: {fps:.2f}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    video_writer.write(frame_resized)
    frame_idx += 1

cap.release()
video_writer.release()
df = pd.DataFrame(alerts_log)
df.to_csv("/content/drive/MyDrive/25_ESD/alerts_log.csv", index=False)
print(df.head(10))
print(f'Output saved to {output_path}')



0: 640x640 1 car, 20.2ms
Speed: 3.3ms preprocess, 20.2ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 1 bicycle, 5 cars, 19.0ms
Speed: 2.9ms preprocess, 19.0ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)


/content/RAFT/core/raft.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/content/RAFT/core/raft.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/content/RAFT/core/raft.py:127: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):



0: 640x640 (no detections), 18.9ms
Speed: 2.8ms preprocess, 18.9ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 3 cars, 18.9ms
Speed: 2.8ms preprocess, 18.9ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 2 cars, 19.5ms
Speed: 2.9ms preprocess, 19.5ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 3 cars, 20.1ms
Speed: 2.8ms preprocess, 20.1ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 3 cars, 1 motorcycle, 19.2ms
Speed: 2.9ms preprocess, 19.2ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 3 cars, 1 motorcycle, 19.1ms
Speed: 2.9ms preprocess, 19.1ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 1 bicycle, 3 cars, 18.8ms
Speed: 2.8ms preprocess, 18.8ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons,